# 04 — Ablations A–E

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/SerreIA/blob/main/notebooks/04_ablations.ipynb)

Les cinq variantes de docs/03-classifieur.md §4 (`configs/ablations/`), entraînées puis
évaluées l'une après l'autre ; tableau final dans `reports/phase1.md`. Les runs terminés
sont enregistrés dans `reports/ablations.json` : une session interrompue reprend à
l'ablation suivante (retirer une entrée pour la relancer). Compter plusieurs heures par
ablation sur un T4.

Sur Colab : *Exécution → Modifier le type d'exécution → GPU (T4)*.

## Paramètres

In [ ]:
ABLATIONS = ["A_baseline", "B_strong_aug", "C_background", "D_ccmt", "E_convnext"]
SET = []  # surcharges appliquées à toutes les ablations
SUBSET = 0
LIMIT_BATCHES = 0
DEVICE = None
DATASETS = ["index", "plantvillage", "ccmt", "plantdoc"]

## Environnement

In [ ]:
# Environnement (PC et Colab) : dépôt, données, sorties (docs/data-drive.md).
# PC : rien n'est installé ; un jeu absent est récupéré du Drive par rclone, sinon la cellule
# s'arrête avec la commande à lancer. Colab : clone du dépôt à REV, installation, données via
# le Drive monté (archives MyDrive/SerreIA/data/*.tar), sorties précédentes restaurées.
import logging
import os
import subprocess
import sys
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/SerreIA"  # Colab : data/*.tar et runs/ ; None : pas de Drive
REPO_URL = "https://github.com/lrivals/SerreIA.git"
REV = "main"
COLAB = "google.colab" in sys.modules

if COLAB:
    ROOT = Path("/content/SerreIA")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[vision]"], check=True
    )
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from tools.notebooks import drive_data  # noqa: E402

if COLAB and DRIVE_DIR:
    drive_data.mount_drive()
    drive_data.restore_all(DRIVE_DIR)
drive_data.ensure(DATASETS, DRIVE_DIR if COLAB else None)

# Colab : tous les cœurs pour le chargement, base MLflow séparée de celle du PC.
ENV_SET = (
    [f"data.num_workers={os.cpu_count()}", f"mlflow.tracking_uri={drive_data.COLAB_MLFLOW}"]
    if COLAB
    else []
)

In [ ]:
# Entraînement, évaluation et Grad-CAM tournent sur le GPU, augmentations comprises.
import torch

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
elif COLAB:
    raise RuntimeError("pas de GPU : Exécution → Modifier le type d'exécution → GPU (T4)")
else:
    print("pas de GPU : tout tournera sur CPU (lent)")

In [ ]:
def ensure_backgrounds(cfg):
    """Fonds de serre requis par le remplacement de fond (docs/03 §3.2)."""
    if not cfg["augment"]["background"]["enabled"]:
        return
    try:
        drive_data.ensure(["backgrounds"], DRIVE_DIR if COLAB else None)
    except RuntimeError as e:
        raise RuntimeError(
            f"{e}\nRemplir data/backgrounds/ puis `tools/get_datasets.sh push backgrounds`, "
            'ou SET = ["augment.background.enabled=false"].'
        ) from None

## Entraînement et évaluation de chaque ablation

In [ ]:
import torch

from vision.eval.ablation_report import load_registry, save_registry
from vision.eval.cross_domain_eval import REPORTS_DIR, log_to_mlflow, run_eval
from vision.train.config import apply_overrides, load_config
from vision.train.train_classifier import pick_device, run

device = pick_device(DEVICE)
registry = load_registry()
for name in ABLATIONS:
    done = registry.get(name)
    if done and (REPORTS_DIR / done / "metrics.json").exists():
        print(f"{name} : déjà fait ({done})")
        continue
    cfg = apply_overrides(
        load_config(ROOT / "configs" / "ablations" / f"{name}.yaml"), ENV_SET + SET
    )
    ensure_backgrounds(cfg)
    print(f"=== {name} ===")
    with drive_data.PeriodicSync(drive_data.OUTPUTS, DRIVE_DIR):
        run_id = run(cfg, device, SUBSET, LIMIT_BATCHES).name
        with torch.inference_mode():
            payload = run_eval(ROOT / "runs" / run_id / "best.pt", device, SUBSET)
        log_to_mlflow(run_id, payload, REPORTS_DIR / run_id)
        registry[name] = run_id
        save_registry(registry)

## Rapport `reports/phase1.md`

In [ ]:
from IPython.display import Markdown, display

from vision.eval.ablation_report import write_phase1_report

report = write_phase1_report({k: v for k, v in registry.items() if k in ABLATIONS})
drive_data.sync_all(DRIVE_DIR)
display(Markdown(report.read_text(encoding="utf-8")))